# 🧠 Milestone 3 — CNN from Scratch (PyTorch)
**Train a Convolutional Neural Network on Mel-Spectrograms**

### What you'll learn:
- PyTorch basics: Tensors, Dataset, DataLoader
- How to convert audio → 2D Mel-Spectrogram images
- How to build a CNN architecture from scratch
- Training loop with loss, optimizer, validation
- Full W&B experiment tracking

### Why CNN works for audio?
Mel-Spectrograms are 2D images (time × frequency). CNNs are great at finding local patterns in images, so they naturally capture rhythmic and timbral patterns in audio!

In [ ]:
!pip install librosa wandb -q
# PyTorch is pre-installed on Kaggle

In [ ]:
import os, random, warnings, time
import numpy as np
import pandas as pd
import librosa
import wandb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Using device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

BASE        = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
STEMS_DIR   = f'{BASE}/genres_stems'
MASHUPS_DIR = f'{BASE}/mashups'
TEST_CSV    = f'{BASE}/test.csv'
GENRES      = ['blues','classical','country','disco','hiphop','jazz','metal','pop','reggae','rock']
ROLL_NO     = 'YOUR_ROLL_NO'  # ⚠️ change this!